# Tiền xử lý dữ liệu Titanic (Preprocessing)

## Improt thư viện

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

## 1. Mục tiêu tiền xử lý 
+ **Mô tả**:
    + Xử lý missing values (Age, Embarked, Cabin, Fare).
    + Feature engineering **nâng cao**: Tạo **Title** từ Name và **FamilySize**.
    + Encoding categorical features (Sex, Embarked, Title).
    + Scaling numerical features (Age, Fare).
+ **Dữ liệu vào**: Từ interim (sau EDA).
+ **Kết quả**: Dữ liệu sạch, lưu vào processed.

## 2. Load dữ liệu và Chuẩn bị

In [2]:
# Load data gốc (train có Survived, test không có)
train_path_raw = '../data/raw/train.csv'
test_path_raw = '../data/raw/test.csv'

df_train = pd.read_csv(train_path_raw)
df_test = pd.read_csv(test_path_raw)

# Tách Survived để xử lý X chung
y_train = df_train['Survived']
df_train.drop('Survived', axis=1, inplace=True)

# Kết hợp để tiền xử lý chung
df_all = pd.concat([df_train, df_test], axis=0, ignore_index=True)

## 3. Feature engineering: Title (Nâng cấp)
+ Trích Mr, Mrs, Miss,... từ Name.
+ Nhóm Title hiếm và Encode thành số.

In [3]:
# Trích Title
df_all['Title'] = df_all['Name'].str.extract(' ([A-Za-z]+)\\.', expand=False)

# Nhóm Title hiếm (tương tự experiment_1.ipynb)
rare_titles = ['Dr', 'Rev', 'Col', 'Major', 'Capt', 'Sir', 'Lady', 'Countess', 'Jonkheer', 'Dona', 'Mme', 'Ms', 'Mlle'] # Bổ sung thêm Dona, Mme, Ms, Mlle
df_all['Title'] = df_all['Title'].replace(['Ms', 'Mlle'], 'Miss')
df_all['Title'] = df_all['Title'].replace(['Mme'], 'Mrs')
df_all['Title'] = df_all['Title'].replace(rare_titles, 'Rare')

# Encode Title thành số
title_mapping = {'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4}
df_all['Title'] = df_all['Title'].map(title_mapping).fillna(4).astype(int)

# Tạo FamilySize
df_all['FamilySize'] = df_all['SibSp'] + df_all['Parch'] + 1

## 4. Xử lý missing values và Drop cột không cần thiết
+ Fill Age và Fare bằng mean (sau khi trích Title, có thể Age đã có giá trị tốt hơn).
+ Fill Embarked bằng mode.
+ Drop Cabin, Ticket, Name, SibSp, Parch, PassengerId.

In [4]:
# Fill Age và Fare bằng mean (sử dụng SimpleImputer)
imputer_num = SimpleImputer(strategy='mean')
df_all[['Age', 'Fare']] = imputer_num.fit_transform(df_all[['Age', 'Fare']])

# Fill Embarked bằng mode
imputer_cat = SimpleImputer(strategy='most_frequent')
df_all['Embarked'] = imputer_cat.fit_transform(df_all[['Embarked']]).ravel()

# Drop các cột không cần thiết (đã lấy Title thay cho Name)
df_all.drop(['Cabin', 'Ticket', 'Name', 'SibSp', 'Parch', 'PassengerId'], axis=1, inplace=True)

# Kiểm tra lại missing
print("Kiểm tra lại missing (sau xử lý):")
print(df_all.isnull().sum())

Kiểm tra lại missing (sau xử lý):
Pclass        0
Sex           0
Age           0
Fare          0
Embarked      0
Title         0
FamilySize    0
dtype: int64


## 5. Encoding categorical features
+ Sex: Binary mapping (0/1).
+ Embarked: One-hot encoding.

In [5]:
# Encoding Sex
df_all['Sex'] = df_all['Sex'].map({'male': 0, 'female': 1})

# One-hot Embarked
df_all = pd.get_dummies(df_all, columns=['Embarked'], prefix='Embarked', drop_first=True)

# Hiển thị 5 dòng đầu
print("\nDataFrame sau encoding:")
print(df_all.head())


DataFrame sau encoding:
   Pclass  Sex   Age     Fare  Title  FamilySize  Embarked_Q  Embarked_S
0       3    0  22.0   7.2500      0           2       False        True
1       1    1  38.0  71.2833      2           2       False       False
2       3    1  26.0   7.9250      1           1       False        True
3       1    1  35.0  53.1000      2           2       False        True
4       3    0  35.0   8.0500      0           1       False        True


## 6. Scaling numerical features
+ Scale Age, Fare, FamilySize để model tốt hơn.

In [6]:
scaler = StandardScaler()
# Sử dụng FamilySize thay cho SibSp/Parch
df_all[['Age', 'Fare', 'FamilySize']] = scaler.fit_transform(df_all[['Age', 'Fare', 'FamilySize']])

print("\nDataFrame sau scaling:")
print(df_all.head())


DataFrame sau scaling:
   Pclass  Sex       Age      Fare  Title  FamilySize  Embarked_Q  Embarked_S
0       3    0 -0.611972 -0.503595      0    0.073352       False        True
1       1    1  0.630431  0.734503      2    0.073352       False       False
2       3    1 -0.301371 -0.490544      1   -0.558346       False        True
3       1    1  0.397481  0.382925      2    0.073352       False        True
4       3    0  0.397481 -0.488127      0   -0.558346       False        True


## 7. Lưu dữ liệu đã xử lý
+ Tách lại train/test và lưu vào processed.

In [7]:
# Lấy kích thước tập train gốc
train_size = len(y_train)

# Tách train và test
train_processed = df_all.iloc[:train_size].copy()
test_processed = df_all.iloc[train_size:].copy()

# Thêm cột Survived vào train
train_processed['Survived'] = y_train.values

# Lưu vào processed
train_processed.to_csv('../data/processed/train_processed.csv', index=False)
test_processed.to_csv('../data/processed/test_processed.csv', index=False)
y_train.to_csv('../data/processed/train_labels.csv', index=False) # Lưu nhãn riêng (giúp experiment_1 chạy được)

print("Data saved to processed/ (train_processed.csv, test_processed.csv, train_labels.csv)")

Data saved to processed/ (train_processed.csv, test_processed.csv, train_labels.csv)


# Kết thúc

In [8]:
# Cách an toàn, đa nền tảng để xóa HTML hiện có và export notebook sang HTML
import os
import subprocess
from pathlib import Path
# Tính toán đường dẫn notebook và output tương đối với file notebook này
nb_dir = Path(__file__).resolve().parent if '__file__' in globals() else Path('.')
# Nếu chạy bên trong notebook, sử dụng thư mục làm việc hiện tại của server notebook
nb_dir = nb_dir if nb_dir.exists() else Path('.')
nb_path = nb_dir / 'preprocessing.ipynb'
out_path = nb_dir / 'preprocessing.html'
# Xóa file output hiện có nếu tồn tại
if out_path.exists():
    print(f'Removing existing file: {out_path}')
    out_path.unlink()
# Chạy nbconvert sử dụng subprocess để ổn định trên Windows
cmd = ['jupyter', 'nbconvert', str(nb_path), '--to', 'html']
print('Running:', ' '.join(cmd))
try:
    subprocess.run(cmd, check=True)
    print('Export complete:', out_path)
except subprocess.CalledProcessError as e:
    print('nbconvert failed with returncode', e.returncode)
    print('Ensure jupyter is available in the PATH of the environment running this notebook.')

Removing existing file: preprocessing.html
Running: jupyter nbconvert preprocessing.ipynb --to html
Export complete: preprocessing.html
